# Rule-Based Sentiment Analysis — Philippine Educational System

**Inputs** (in `datasets/` locally, or uploaded to `/content/` in Colab):
- `final_training_data.csv` and `final_unseen_data.csv` — run `DatasetFixes.ipynb` first
- `final_manual_labels.csv` — the manual labels (`manual_label`: `1` positive, `0` neutral, `-1` negative), matched by `record_id`

**What it does:** compares three versions of the rules with 10-fold cross-validation on the labeled training records (v1 lexicon only → v2 + negation → v3 + negation + intensifiers), picks the version with the best macro-F1, then tests it once on the unseen 10%.

**Outputs:** `predictions_training.csv` and `predictions_unseen.csv` (record_id, manual label, score, prediction) for the sentiment-per-topic step.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Looks in datasets/ (running locally from the repo) and /content/ (Colab uploads).
# Also finds Colab-renamed copies like "final_training_data (1).csv".
DATA_DIRS = ["datasets", "/content/datasets", "/content", "."]

def find_file(name, required=True):
    stem, ext = os.path.splitext(name)
    for d in DATA_DIRS:
        exact = os.path.join(d, name)
        if os.path.exists(exact):
            return exact
        hits = sorted(glob.glob(os.path.join(d, stem + " (*)" + ext)))
        if hits:
            return hits[-1]
    if required:
        raise FileNotFoundError(f"Upload {name} to /content/ (Colab) or keep it in datasets/ (local).")
    return None

if IN_COLAB and not find_file("final_training_data.csv", required=False):
    uploaded = files.upload()   # upload the training, unseen, and manual label files

TRAIN_FILE = find_file("final_training_data.csv")
TEST_FILE  = find_file("final_unseen_data.csv")
LABEL_FILE = find_file("final_manual_labels.csv", required=False)
print("Using:", TRAIN_FILE, "|", TEST_FILE, "|", LABEL_FILE)

In [ ]:
INCLUDE_ARTICLES = True
LANG_COL = "detected_language"
FINAL_VERSION = "v3_+negation+intensity"   # used only if there are no training labels
CLASSES = ["positive", "neutral", "negative"]          # 3 separate classes
COLORS = {"positive": "#2e9e5b", "neutral": "#8a8f98", "negative": "#d64545"}
LABEL_MAP = {"1": "positive", "1.0": "positive", "0": "neutral", "0.0": "neutral",
             "-1": "negative", "-1.0": "negative",
             "positive": "positive", "neutral": "neutral", "negative": "negative"}

labels = None
if LABEL_FILE:
    labels = pd.read_csv(LABEL_FILE)[["record_id", "manual_label"]]
    labels["label"] = labels["manual_label"].astype(str).str.strip().str.lower().map(LABEL_MAP)
    labels = labels.dropna(subset=["label"]).drop_duplicates("record_id")[["record_id", "label"]]

def load(path):
    d = pd.read_csv(path)
    d["text"] = d["text"].where(d["text"].notna() & (d["text"].astype(str).str.strip() != ""), d["title"])
    d = d.dropna(subset=["text"])
    d = d[d["text"].str.strip() != ""]
    if not INCLUDE_ARTICLES and "record_type" in d.columns:
        d = d[d["record_type"] != "article"]
    if labels is not None:
        d = d.merge(labels, on="record_id", how="left")
    else:
        d["label"] = np.nan
    return d.reset_index(drop=True)

dev_df, test_df = load(TRAIN_FILE), load(TEST_FILE)
dev_lab, test_lab = dev_df.dropna(subset=["label"]), test_df.dropna(subset=["label"])
HAS_TRAIN_LABELS = len(dev_lab) >= 30
HAS_TEST_LABELS = len(test_lab) > 0

print(f"Training rows: {len(dev_df)} ({len(dev_lab)} labeled) | Unseen rows: {len(test_df)} ({len(test_lab)} labeled)")
for name, d in (("training", dev_df), ("unseen", test_df)):
    missing = d.loc[d["label"].isna(), "record_id"].tolist()
    if missing:
        print(f"  {len(missing)} {name} records have no manual label and are left out of the metrics: {missing}")

In [ ]:
POS_EN = """good great excellent love loved helpful improve improved improving improvement better amazing best
thanks thank support supported proud safe benefit benefits progress success successful effective wonderful
awesome nice happy glad hope hopeful congratulations congrats recommend valuable quality decent strong
brilliant impressive agree perfect grateful appreciate appreciated deserve opportunity empower inspiring
honest handy correct excited free fair solid smart worth helpful enjoy enjoyed""".split()
NEG_EN = """bad poor terrible hate worse worst fail failed failure useless problem problems lack lacking crisis
corrupt corruption expensive stress stressful unfair broken shortage faulty sad tough struggle struggling
struggled difficult bankrupt deteriorating deteriorate poverty blame blaming limited mediocre downside
ignorant stupid dumb ridiculous shame shameful disappointing disappointed unfortunately issues suffer
suffering victim violence shooting danger dangerous unsafe worry worried angry annoying waste wasted
overpriced outdated flawed incompetent lazy sucks awful horrible pathetic toxic abuse cheating cheat scam
fake bully bullying unemployed underpaid overworked burnout inadequate insufficient neglect weak wrong
sick decline declining lowest harder""".split()
POS_TL = """maganda ganda mahusay husay magaling galing salamat pasalamat mabuti masaya saya tama maayos ayos
makakabuti pabor mabait bait matagumpay tagumpay malinaw linaw tulong bilib hanga mabuhay maasahan
pakinabang swerte sulit tuwa malasakit pagmamahal""".split()
NEG_TL = """pangit masama mahirap hirap kulang palpak sayang mali pagod reklamo kakulangan bulok magnanakaw
nakawan nakaw ninakaw tamad bobo tanga gago buwisit inis galit nakakainis nakakagalit hiya kahihiyan
kawawa pahirap pabaya kapabayaan mababa bumagsak bagsak mahina kahirapan problema suliranin trahedya
sira delikado panganib takot lungkot malungkot sakit iyak abuso korap kurakot kurapsyon kapos gutom
bumaba""".split()

NEGATORS = {"not","no","never","without","hardly","barely","nothing","hindi","di","wala","walang","ayaw","hinde"}
INTENSIFIERS = {"very","so","really","extremely","super","sobra","talaga","grabe","napaka","sobrang"}
POS_ALL, NEG_ALL = set(POS_EN) | set(POS_TL), set(NEG_EN) | set(NEG_TL)
PREFIXES = ["napaka","pinaka","nakaka","magpa","nagpa","mapag","pagka","mag","nag","pag","ma","na","ka","pa"]
SUFFIXES = ["han","hin","an","in"]

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|@\w+", " ", text)
    text = re.sub(r"\bud[0-9a-f]{2,4}\b", " ", text)       # emoji escape leftovers like ud83c
    text = re.sub(r"problem[- ]solving", " ", text)           # "problem solving" is not a complaint
    return re.findall(r"[a-zñ']+", text)

def word_polarity(tok, lex_pos, lex_neg):
    """+1 / -1 / 0. Also tries Tagalog affix stripping (mahirap -> hirap, nahirapan -> hirap)."""
    if tok in lex_pos: return 1.0
    if tok in lex_neg: return -1.0
    cands = set()
    for p in PREFIXES:
        if tok.startswith(p) and len(tok) - len(p) >= 4:
            rest = tok[len(p):]
            cands.add(rest)
            for s in SUFFIXES:
                if rest.endswith(s) and len(rest) - len(s) >= 4:
                    cands.add(rest[:-len(s)])
    for c in cands:
        if c in lex_pos: return 1.0
        if c in lex_neg: return -1.0
    return 0.0

class RuleBasedSentiment:
    """score > band -> positive | score < -band -> negative | otherwise -> neutral"""
    def __init__(self, use_negation=False, use_intensifiers=False, neutral_band=0.0):
        self.use_negation, self.use_intensifiers = use_negation, use_intensifiers
        self.neutral_band = neutral_band

    def score(self, text):
        toks, total = tokenize(text), 0.0
        for i, t in enumerate(toks):
            s = word_polarity(t, POS_ALL, NEG_ALL)
            if s == 0.0:
                continue
            window = toks[max(0, i-3):i]
            if self.use_intensifiers and any(w in INTENSIFIERS for w in window):
                s *= 1.5
            if self.use_negation and any(w in NEGATORS for w in window):
                s *= -1
            total += s
        return total

    def label_from_score(self, sc):
        return ("positive" if sc > self.neutral_band
                else "negative" if sc < -self.neutral_band else "neutral")

    def fit(self, X, y=None): return self
    def predict(self, X): return np.array([self.label_from_score(self.score(t)) for t in X])

VERSIONS = {
    "v1_lexicon_only":        dict(),
    "v2_+negation":           dict(use_negation=True),
    "v3_+negation+intensity": dict(use_negation=True, use_intensifiers=True),
}

In [ ]:
def plot_distribution(datasets, title):
    fig, ax = plt.subplots(figsize=(7, 4))
    w, x = 0.8 / len(datasets), np.arange(len(CLASSES))
    for k, (name, preds) in enumerate(datasets.items()):
        pct = pd.Series(preds).value_counts(normalize=True).reindex(CLASSES, fill_value=0) * 100
        bars = ax.bar(x + k*w, pct.values, w, label=name, alpha=0.9 - 0.3*k)
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f"{b.get_height():.1f}%",
                    ha="center", fontsize=8)
    ax.set_xticks(x + w*(len(datasets)-1)/2); ax.set_xticklabels(CLASSES)
    ax.set_ylabel("% of records"); ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

def plot_score_hist(df, title, score_col="sentiment_score", hue_col="pred"):
    fig, ax = plt.subplots(figsize=(8, 4))
    lo, hi = int(df[score_col].min()) - 1, int(df[score_col].max()) + 2
    bins = np.arange(lo - 0.5, hi + 0.5, 1)
    for c in CLASSES:
        ax.hist(df.loc[df[hue_col] == c, score_col], bins=bins, color=COLORS[c],
                alpha=0.85, label=c)
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("sentiment score (sum of word scores)"); ax.set_ylabel("number of records")
    ax.set_title(title); ax.legend(); plt.tight_layout(); plt.show()

def plot_stacked(df, col, title):
    if col not in df.columns: return
    ct = pd.crosstab(df[col], df["pred"], normalize="index").reindex(columns=CLASSES, fill_value=0) * 100
    ax = ct.plot(kind="bar", stacked=True, figsize=(7, 4), color=[COLORS[c] for c in CLASSES])
    ax.set_ylabel("% of records"); ax.set_title(title); ax.set_xlabel("")
    plt.xticks(rotation=0); plt.legend(title="")
    plt.tight_layout(); plt.show()

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES)
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12)
    plt.colorbar(im); plt.tight_layout(); plt.show()

def plot_class_metrics(y_true, y_pred, title):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    x, w = np.arange(3), 0.27
    fig, ax = plt.subplots(figsize=(7, 4))
    for k, (vals, name) in enumerate([(p, "precision"), (r, "recall"), (f, "F1")]):
        bars = ax.bar(x + (k-1)*w, vals, w, label=name)
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f"{b.get_height():.2f}",
                    ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES); ax.set_ylim(0, 1.1)
    ax.set_title(title); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
def cross_validate(params, data, n_splits=10):
    X, y = data["text"].values, data["label"].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs, f1s = [], []
    for tr, va in skf.split(X, y):
        pred = RuleBasedSentiment(**params).fit(X[tr], y[tr]).predict(X[va])
        accs.append(accuracy_score(y[va], pred))
        f1s.append(precision_recall_fscore_support(y[va], pred, average="macro", zero_division=0)[2])
    return np.mean(accs), np.std(accs), np.mean(f1s)

In [ ]:
# ---------------- 1. Choose the rule version (10-fold CV on labeled training records) ----------------
if HAS_TRAIN_LABELS:
    n_splits = max(2, min(10, int(dev_lab["label"].value_counts().min())))
    print("Label distribution (training):\n", dev_lab["label"].value_counts(), "\n")

    results = []
    for name, params in VERSIONS.items():
        acc, sd, f1 = cross_validate(params, dev_lab, n_splits)
        results.append((name, acc, sd, f1))
        print(f"{name:26s} {n_splits}-fold CV acc={acc:.4f} (+/- {sd:.4f})  macro-F1={f1:.4f}")
    best_name = max(results, key=lambda r: r[3])[0]
    print("\nSelected:", best_name)

    names = [r[0] for r in results]; x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x-0.2, [r[1] for r in results], 0.4, yerr=[r[2] for r in results], capsize=4, label="CV accuracy")
    ax.bar(x+0.2, [r[3] for r in results], 0.4, label="CV macro-F1")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15); ax.set_ylim(0, 1)
    ax.set_title("Model versions: cross-validation metrics"); ax.legend()
    plt.tight_layout(); plt.show()
else:
    best_name = FINAL_VERSION
    print("Not enough training labels for cross-validation; using", best_name)
    preds = {name: RuleBasedSentiment(**params).predict(dev_df["text"].values)
             for name, params in VERSIONS.items()}
    for name, p in preds.items():
        print(f"{name:26s}", pd.Series(p).value_counts(normalize=True).round(3).to_dict())
    plot_distribution(preds, "Training: sentiment split by model version")

# ---------------- 2. Score every record with the selected version ----------------
final = RuleBasedSentiment(**VERSIONS[best_name])
for d in (dev_df, test_df):
    d["sentiment_score"] = [final.score(t) for t in d["text"]]
    d["pred"] = final.predict(d["text"].values)

# ---------------- 3. Final test on the unseen 10% ----------------
if HAS_TEST_LABELS:
    t = test_df.dropna(subset=["label"])
    print(f"\nUNSEEN accuracy ({len(t)} labeled records):", round(accuracy_score(t["label"], t["pred"]), 4))
    print(classification_report(t["label"], t["pred"], labels=CLASSES, zero_division=0))
    plot_confusion(t["label"], t["pred"], "Confusion matrix (unseen)")
    plot_class_metrics(t["label"], t["pred"], "Per-class metrics (unseen)")
    plot_distribution({"actual": t["label"].values, "predicted": t["pred"].values},
                      "Actual vs predicted (unseen)")
    plot_score_hist(t, "Score distribution by TRUE label (unseen)", hue_col="label")
    plot_stacked(t, LANG_COL, "Predictions by language (unseen)")
else:
    print("\nThe unseen records have no manual labels yet, so the final test cannot be scored.")
    print("Label them (see records_to_label.csv from DatasetFixes.ipynb) and re-run.")
    plot_distribution({"training": dev_df["pred"].values, "unseen": test_df["pred"].values},
                      "Training vs unseen: predicted sentiment split")

for name, d in (("TRAINING", dev_df), ("UNSEEN", test_df)):
    print(f"\n===== {name} (n={len(d)}) using {best_name} =====")
    print(d["pred"].value_counts().reindex(CLASSES, fill_value=0))
    plot_stacked(d, "platform", f"{name}: sentiment by platform")

if HAS_TRAIN_LABELS:
    errors = dev_df.dropna(subset=["label"]).query("label != pred")
    print("\nError sample (training):")
    print(errors.sample(min(15, len(errors)), random_state=1)[["text", "label", "pred"]])

# ---------------- 4. Save predictions for sentiment per topic ----------------
keep = [c for c in ["record_id", "platform", "record_type", LANG_COL, "text", "label", "sentiment_score", "pred"]
        if c in dev_df.columns]
dev_df[keep].to_csv("predictions_training.csv", index=False)
test_df[keep].to_csv("predictions_unseen.csv", index=False)
print("\nSaved: predictions_training.csv, predictions_unseen.csv")

In [ ]:
if IN_COLAB:
    files.download("predictions_training.csv")
    files.download("predictions_unseen.csv")